# Lab 9.3 &mdash; The Manifest Is the Deployment

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 9 &mdash; Deployment &amp; AgentOps**

### What you'll do
- Turn the production-readiness checklist into rules that run
- Lint a manifest that looks fine and find five reasons it is not
- Separate what blocks a release from what is worth an argument
- Send the object you linted to a real API server

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, and none of them needs a cluster, so your
> score never depends on a live endpoint or on `kubectl` working. Cells marked **Run it for real**
> do call the sandbox model or your namespace; if either is unreachable they print how to fix it
> instead of crashing.

> **A checklist you have to remember is a checklist you will not run.** Everything in
> this lab is a predicate over a Python dict, and the dict is exactly what `kubectl`
> receives &mdash; a Kubernetes manifest is JSON, and YAML is a surface syntax over it.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-9-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- your own namespace --------------------------------------------------
# You deploy into your own namespace, published at your own host. Both are injected into
# the sandbox, so nothing here is hardcoded and nothing here needs them to be set.
#
# Read ONLY from APP_NAMESPACE, never derived from the hostname. A cell below runs
# kubectl against whatever this says, and a namespace guessed from a machine name is
# the wrong thing to point kubectl at.
APP_NS   = os.environ.get("APP_NAMESPACE", "")
APP_HOST = os.environ.get("APP_HOST", "")

print("work dir :", WORK)
print("model    :", LLM_MODEL or "(not configured -- graded cells still work)")
print("namespace:", APP_NS or "(unknown -- graded cells still work)")

In [ ]:
# ------------------------------------------------- two manifests, as objects
# A Kubernetes manifest IS JSON. YAML is a surface syntax over it, and `kubectl apply`
# accepts either -- which is why everything below is stdlib, exact, and needs no cluster
# until you decide to send it to one.

FLAWED = [
    {"apiVersion": "apps/v1", "kind": "Deployment",
     "metadata": {"name": "agent-app"},
     "spec": {
        "replicas": 1,
        "selector": {"matchLabels": {"app": "agent-app"}},
        "template": {"metadata": {"labels": {"app": "agent-app"}},
          "spec": {"containers": [{
             "name": "app",
             "image": "registry.internal/agent-app:latest",
             "ports": [{"containerPort": 8000}],
             "env": [
                {"name": "LAB_LLM_BASE_URL", "value": "http://gateway.llm-serving:8080/v1"},
                {"name": "OPENAI_API_KEY",   "value": "sk-live-EXAMPLE-0000000000"},
             ],
             "livenessProbe":  {"httpGet": {"path": "/healthz", "port": 8000},
                                "periodSeconds": 20},
             "readinessProbe": {"httpGet": {"path": "/healthz", "port": 8000},
                                "periodSeconds": 10},
          }]}}}},
    {"apiVersion": "v1", "kind": "Service",
     "metadata": {"name": "agent-app"},
     "spec": {"selector": {"app": "agent-app"},
              "ports": [{"port": 80, "targetPort": 8000}]}},
]

FIXED = [
    {"apiVersion": "apps/v1", "kind": "Deployment",
     "metadata": {"name": "agent-app"},
     "spec": {
        "replicas": 1,
        "selector": {"matchLabels": {"app": "agent-app"}},
        "template": {"metadata": {"labels": {"app": "agent-app"}},
          "spec": {
            "securityContext": {"runAsNonRoot": True, "runAsUser": 1000},
            "containers": [{
             "name": "app",
             "image": "registry.internal/agent-app:v3",
             "ports": [{"containerPort": 8000}],
             "envFrom": [{"secretRef": {"name": "llm-credentials"}}],
             "env": [{"name": "LOG_LEVEL", "value": "info"}],
             "livenessProbe":  {"httpGet": {"path": "/healthz", "port": 8000},
                                "initialDelaySeconds": 15, "periodSeconds": 20},
             "readinessProbe": {"httpGet": {"path": "/readyz", "port": 8000},
                                "initialDelaySeconds": 5, "periodSeconds": 10},
             "resources": {"requests": {"cpu": "100m", "memory": "128Mi"},
                           "limits":   {"cpu": "500m", "memory": "512Mi"}},
          }]}}}},
    {"apiVersion": "v1", "kind": "Service",
     "metadata": {"name": "agent-app"},
     "spec": {"selector": {"app": "agent-app"},
              "ports": [{"port": 80, "targetPort": 8000}]}},
    {"apiVersion": "networking.k8s.io/v1", "kind": "Ingress",
     "metadata": {"name": "agent-app"},
     "spec": {"ingressClassName": "nginx",
              "rules": [{"host": "REPLACED-AT-RENDER-TIME",
                         "http": {"paths": [{"path": "/", "pathType": "Prefix",
                                             "backend": {"service": {"name": "agent-app",
                                                                     "port": {"number": 80}}}}]}}]}},
    {"apiVersion": "autoscaling/v2", "kind": "HorizontalPodAutoscaler",
     "metadata": {"name": "agent-app"},
     "spec": {"scaleTargetRef": {"apiVersion": "apps/v1", "kind": "Deployment",
                                 "name": "agent-app"},
              "minReplicas": 1, "maxReplicas": 3,
              "metrics": [{"type": "Resource",
                           "resource": {"name": "cpu",
                                        "target": {"type": "Utilization",
                                                   "averageUtilization": 70}}}]}},
]

def containers(doc):
    """Every container in a Deployment, or nothing for any other kind."""
    if doc.get("kind") != "Deployment":
        return []
    return doc["spec"]["template"]["spec"]["containers"]

print(f"FLAWED: {len(FLAWED)} objects   FIXED: {len(FIXED)} objects")

## Concept

Every deployment guide ends with a checklist: resource limits, probes, no secrets in the image,
more than one replica. Written as prose it is a thing to forget. Written as five predicates over
the manifest it is a thing that runs in CI and fails a pull request.

You are about to find out that the interesting part is not writing the rules. It is deciding
which findings block a release, and noticing that one of your rules is wrong for this workload.

## Section 1 &mdash; The checklist as predicates

Each rule takes one object and the whole document set, and returns a list of findings.
Findings are data, not printed text, so the same rules can fail a build and render a report.

In [ ]:
SECRETISH = ("KEY", "TOKEN", "SECRET", "PASSWORD", "CREDENTIAL")

def finding(rule, severity, doc, detail):
    return {"rule": rule, "severity": severity,
            "object": f"{doc.get('kind')}/{doc.get('metadata', {}).get('name')}",
            "detail": detail}


def rule_resources(doc, docs):
    """Every container states what it needs and what it may take."""
    out = []
    for c in containers(doc):
        res = c.get("resources", {})
        # TODO: a container needs BOTH -- requests, so the scheduler can place it, and
        # limits, so one bad request cannot take the node down. Flag a container missing
        # either one.
        if BLANK:
            out.append(finding("resources", "block", doc,
                               f"container {c['name']} is missing requests and/or limits"))
    return out


def rule_probes(doc, docs):
    """Liveness and readiness must exist, and must not be the same check."""
    out = []
    for c in containers(doc):
        live, ready = c.get("livenessProbe"), c.get("readinessProbe")
        if not live or not ready:
            out.append(finding("probes", "block", doc,
                               f"container {c['name']} is missing a probe"))
            continue
        # TODO: two probes pointed at the same path answer the same question twice.
        # Compare where each one actually points.
        if BLANK:
            out.append(finding("probes", "block", doc,
                               f"container {c['name']}: liveness and readiness both probe "
                               f"{live['httpGet']['path']}"))
    return out


def rule_literal_secret(doc, docs):
    """Credentials arrive from a Secret, never as a literal in the manifest."""
    out = []
    for c in containers(doc):
        for e in c.get("env", []):
            # TODO: an env entry is a literal if it carries a "value" rather than a
            # "valueFrom". Flag the ones whose NAME looks like a credential.
            if BLANK:
                out.append(finding("literal-secret", "block", doc,
                                   f"container {c['name']}: {e['name']} is a literal value "
                                   f"in the manifest"))
    return out

In [ ]:
# Three more rules, written for you. Read them: the last one is the one to argue about.

def rule_image_tag(doc, docs):
    """An image without an explicit, immutable tag is a deployment you cannot reproduce."""
    out = []
    for c in containers(doc):
        tag = c["image"].rsplit(":", 1)[-1] if ":" in c["image"].rsplit("/", 1)[-1] else ""
        if tag in ("", "latest"):
            out.append(finding("image-tag", "block", doc,
                               f"container {c['name']} uses {c['image']!r} -- "
                               f"two rollouts of this are not the same deployment"))
    return out


def rule_capacity(doc, docs):
    """One replica is a single point of failure, unless something can add more."""
    if doc.get("kind") != "Deployment":
        return []
    name = doc["metadata"]["name"]
    has_hpa = any(d.get("kind") == "HorizontalPodAutoscaler"
                  and d["spec"]["scaleTargetRef"]["name"] == name for d in docs)
    if doc["spec"].get("replicas", 1) < 2 and not has_hpa:
        return [finding("capacity", "block", doc,
                        "one replica and nothing that can add another -- a rollout is an outage")]
    return []


def rule_hpa_signal(doc, docs):
    """An advisory, and the most interesting rule here. See Section 3."""
    if doc.get("kind") != "HorizontalPodAutoscaler":
        return []
    names = [m.get("resource", {}).get("name") for m in doc["spec"].get("metrics", [])]
    if names == ["cpu"]:
        return [finding("hpa-signal", "advise", doc,
                        "scales on CPU only -- check that CPU actually tracks load for this "
                        "workload before relying on it")]
    return []


RULES = [rule_resources, rule_probes, rule_literal_secret,
         rule_image_tag, rule_capacity, rule_hpa_signal]


def lint(docs, rules=None):
    """Every finding across every object, in rule order."""
    return [f for doc in docs for rule in (rules or RULES) for f in rule(doc, docs)]


def blocking(findings):
    return [f for f in findings if f["severity"] == "block"]

In [ ]:
# --- Self-check: Section 1
check("the flawed manifest has no resource requests or limits",
      lambda: any(f["rule"] == "resources" for f in lint(FLAWED)))
check("...and both its probes ask the same question",
      lambda: any(f["rule"] == "probes" for f in lint(FLAWED)))
check("...and it carries a live API key as a literal",
      lambda: any(f["rule"] == "literal-secret" for f in lint(FLAWED)),
      "which is now in git, in the image, and in every `kubectl get deploy -o yaml`")
check("...and it deploys :latest",
      lambda: any(f["rule"] == "image-tag" for f in lint(FLAWED)))
check("...and one replica with nothing to add another",
      lambda: any(f["rule"] == "capacity" for f in lint(FLAWED)))
check("FIVE blocking findings in a manifest that looks perfectly ordinary",
      lambda: len(blocking(lint(FLAWED))) == 5)
check("the fixed manifest has none of them",
      lambda: len(blocking(lint(FIXED))) == 0)
check("the literal-secret rule does not flag an ordinary variable",
      lambda: not any("LOG_LEVEL" in f["detail"] for f in lint(FIXED)),
      "a rule that flags everything gets switched off in a week")
check("the fixed manifest still has one thing to say",
      lambda: len(lint(FIXED)) == 1)

def _report():
    for label, docs in (("FLAWED", FLAWED), ("FIXED", FIXED)):
        fs = lint(docs)
        print(f"  {label}: {len(blocking(fs))} blocking, {len(fs) - len(blocking(fs))} advisory")
        for f in fs:
            print(f"    [{f['severity']:6}] {f['rule']:15} {f['object']:35} {f['detail'][:60]}")
guard(_report)

## Section 2 &mdash; What a finding costs

A linter that only prints is a linter people ignore. The value is in the two decisions attached
to each rule: does it fail the build, and can it be waived?

In [ ]:
def gate(docs, waivers=()) -> dict:
    """The release decision. Blocking findings stop it unless explicitly waived."""
    findings = lint(docs)
    blocked = [f for f in blocking(findings) if f["rule"] not in waivers]
    waived  = [f for f in blocking(findings) if f["rule"] in waivers]
    return {"pass": not blocked,
            "blocked_by": sorted({f["rule"] for f in blocked}),
            "waived": sorted({f["rule"] for f in waived}),
            "advisories": [f["rule"] for f in findings if f["severity"] == "advise"]}

In [ ]:
# --- Self-check: Section 2
check("the flawed manifest does not ship",
      lambda: gate(FLAWED)["pass"] is False)
check("and the gate says exactly which rules stopped it",
      lambda: gate(FLAWED)["blocked_by"]
              == ["capacity", "image-tag", "literal-secret", "probes", "resources"])
check("the fixed manifest ships",
      lambda: gate(FIXED)["pass"] is True)
check("an advisory never blocks",
      lambda: gate(FIXED)["advisories"] == ["hpa-signal"] and gate(FIXED)["pass"] is True)
check("a waiver is recorded, not silent",
      lambda: gate(FLAWED, waivers=("capacity",))["waived"] == ["capacity"])
check("waiving one rule does not ship a manifest that fails four others",
      lambda: gate(FLAWED, waivers=("capacity",))["pass"] is False,
      "the usual failure of a checklist is that one waiver becomes a blanket one")
check("waiving everything ships anything, which is why waivers need a name on them",
      lambda: gate(FLAWED, waivers=tuple(gate(FLAWED)["blocked_by"]))["pass"] is True)

## Section 3 &mdash; The rule that is wrong

`rule_hpa_signal` is an advisory rather than a block, and it is the only rule here that is
about *this* workload rather than about deployments in general.

Autoscaling on CPU is the default because for most web services CPU is load: more requests, more
parsing, rendering and serialising, more CPU. An agent service does almost none of that. It sends
a request to a gateway and waits, and waiting consumes no CPU at all.

In [ ]:
def cpu_under_load(concurrent: int, call_seconds: float = 8.0,
                   cpu_seconds_per_request: float = 0.015) -> float:
    """CPU utilisation of one replica serving `concurrent` IO-bound agent requests.

    Each request spends call_seconds waiting on the gateway and cpu_seconds_per_request
    actually running code -- parsing JSON, building the prompt, formatting the answer.
    """
    busy = concurrent * cpu_seconds_per_request
    return 100.0 * busy / call_seconds


def hpa_would_scale(utilisation: float, target: int = 70) -> bool:
    return utilisation > target

In [ ]:
# --- Self-check: Section 3
check("one request in flight is invisible to the CPU metric",
      lambda: cpu_under_load(1) < 1)
check("forty concurrent requests are still under 10% CPU",
      lambda: cpu_under_load(40) < 10)
check("...so an HPA targeting 70% CPU does not scale",
      lambda: hpa_would_scale(cpu_under_load(40)) is False)
check("nor at a hundred and twenty",
      lambda: hpa_would_scale(cpu_under_load(120)) is False)
check("it finally crosses 70% at four hundred concurrent on one replica",
      lambda: hpa_would_scale(cpu_under_load(400)) is True,
      "a concurrency at which every request has been queueing for minutes -- the "
      "autoscaler fires long after the callers gave up")
check("the CPU metric only moves for work the agent does not do",
      lambda: hpa_would_scale(cpu_under_load(40, cpu_seconds_per_request=1.5)) is True)

def _cpu():
    print(f"  {'concurrent':>11} {'CPU %':>7} {'HPA scales?':>12}")
    for n in (1, 10, 40, 100, 400):
        u = cpu_under_load(n)
        print(f"  {n:>11} {u:>6.1f}% {str(hpa_would_scale(u)):>12}")
guard(_cpu)

### Read it

The HPA is correctly configured, correctly deployed, and will never fire. Latency will go to
forty seconds and the dashboard will show a replica that is 4% busy.

Two things follow.

1. **The advisory is right and the rule cannot be a block**, because the same HPA is exactly
   right for a service that renders templates. A checklist encodes assumptions about the
   workload; this one names the assumption instead of hiding it.
2. **The signal has to be something that grows with load.** In-flight requests, queue depth, or
   time-to-first-token. Lab 9.5 picks one and tests it.

The starter manifest shipped with this module has this HPA in it, on purpose. It demonstrates the
object, and it is the wrong signal for the workload &mdash; which is a more useful thing for you
to have found here than to discover on a Monday.

## Run it for real

Your linted manifest, sent to the real API server with `--dry-run=server`. That runs
authentication, RBAC, admission control, quota and schema validation, and changes nothing.

In [ ]:
import shutil, subprocess

def render(docs, namespace: str, host: str = "") -> str:
    """Write the objects to a JSON file kubectl can apply. Nothing is templated by hand."""
    out = []
    for d in docs:
        d = json.loads(json.dumps(d))                 # a copy; never mutate the source
        d.setdefault("metadata", {})["namespace"] = namespace
        if d["kind"] == "Ingress" and host:
            d["spec"]["rules"][0]["host"] = host
        out.append(d)
    path = os.path.join(WORK, "agent-app.json")
    with open(path, "w") as fh:
        json.dump({"apiVersion": "v1", "kind": "List", "items": out}, fh, indent=1)
    return path


def _dry_run():
    if not APP_NS:
        print("APP_NAMESPACE is not set, so there is nothing safe to point kubectl at.")
        print("In a sandbox terminal it is already exported; check with `env | grep APP_`.")
        print("Your namespace is your pod name without the trailing -0.")
        return
    if not shutil.which("kubectl"):
        print("kubectl is not on PATH in this kernel -- open a terminal in the sandbox instead.")
        return
    if not gate(FIXED)["pass"]:
        print("The gate says no. Fix the findings before deploying.")
        return
    path = render(FIXED, APP_NS, APP_HOST or f"{APP_NS}-app.example")
    print("wrote", path)
    r = subprocess.run(["kubectl", "apply", "-n", APP_NS, "--dry-run=server", "-f", path],
                       capture_output=True, text=True, timeout=60)
    print(r.stdout.strip() or r.stderr.strip()[:600])
    print("\nNothing was created. To deploy for real, in a sandbox TERMINAL:")
    print(f"  kubectl apply -n {APP_NS} -f {path}")
    print(f"  kubectl get pods,svc,ingress,hpa -n {APP_NS}")

guard(_dry_run)

### Read it

Whatever the dry run said, notice which failures it can find and which it cannot. It validates
the schema, your RBAC, the namespace quota and every admission webhook &mdash; and it says
nothing at all about whether your probes are the right way round, whether the image exists, or
whether the HPA will ever fire.

That is the division of labour: the API server checks that the object is legal, and your linter
checks that it is a good idea.

A full, commented starter manifest &mdash; the same objects with the Ingress, the Secret
references and the security context filled in &mdash; ships beside this notebook as
`app-deploy-example.yaml`. Use it for the capstone.

In [ ]:
score()

## Your turn

1. Add the rule that catches the thing this lab did not: a `Secret` referenced by `envFrom` that
   does not exist in the namespace. Note that it cannot be checked offline, and decide where it
   belongs &mdash; the linter, the dry run, or readiness.
2. Write the waiver format. A waiver needs a rule, a reason, an owner and an expiry, or it is a
   permanent silence with a comment on it.
3. Run the linter over the starter manifest shipped with this module. It should pass. If it does
   not, one of you is wrong and it is worth finding out which.